In [ ]:
# =====================================================================
# MODEL: Recurrent Neural Network (LSTM)
# DATASET: data_without_weather.csv
# GOAL: Predict 'PROBABLE_CAUSE' (Classification)
# =====================================================================
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, LSTM
from tensorflow.keras.callbacks import EarlyStopping

# 1. Load Data
df = pd.read_csv('data_without_weather.csv')
df = df.dropna(subset=['PROBABLE_CAUSE'])

# Filter rare classes (< 5 samples)
class_counts = df['PROBABLE_CAUSE'].value_counts()
valid_classes = class_counts[class_counts >= 5].index
df = df[df['PROBABLE_CAUSE'].isin(valid_classes)]

cols_to_drop = ['PROBABLE_CAUSE', 'DATEOFDTC', 'DTC_NO_JOB_NO', 'OBSERVATION_DURING_DTC', 'PLACE_OF_DAMAGED', 'FEEDER']
X = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
y_text = df['PROBABLE_CAUSE']

# 2. Target Encoding
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_text)
num_classes = len(label_encoder.classes_)

# 3. Preprocessing Pipeline
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X.select_dtypes(include=['number']).columns.tolist()

preprocessor = ColumnTransformer(transformers=[
    ("num", Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_cols),
    ("cat", Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='missing')), ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_cols)
])

# 4. Train/Test Split & Transformation
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)

# ** CRITICAL STEP FOR RNN: Reshape 2D to 3D (samples, time_steps=1, features) **
X_train_rnn = X_train_prep.reshape((X_train_prep.shape[0], 1, X_train_prep.shape[1]))
X_test_rnn = X_test_prep.reshape((X_test_prep.shape[0], 1, X_test_prep.shape[1]))

# 5. Build RNN (LSTM) Model
rnn_model = Sequential([
    LSTM(64, return_sequences=False, input_shape=(1, X_train_prep.shape[1])),
    BatchNormalization(), Dropout(0.3),
    Dense(32, activation="relu"),
    BatchNormalization(), Dropout(0.3),
    Dense(num_classes, activation='softmax')
])

rnn_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# 6. Train Model
print("Training RNN (LSTM) on Data WITHOUT Weather...")
rnn_model.fit(X_train_rnn, y_train, epochs=30, batch_size=64, validation_split=0.2, callbacks=[early_stopping], verbose=1)

# 7. Evaluate
y_pred_probs = rnn_model.predict(X_test_rnn)
y_pred = np.argmax(y_pred_probs, axis=1)

print("\n--- RNN RESULTS (NO WEATHER) ---")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Macro Avg F1-Score:", f1_score(y_test, y_pred, average='macro'))
print("Weighted Avg F1-Score:", f1_score(y_test, y_pred, average='weighted'))
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=label_encoder.classes_, zero_division=0))